# L69HYJ4Y3TR91

Packages

In [ ]:
%store -r sales_data_merged

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
from pathlib import Path
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.arima.model import ARIMA
import tabulate
from IPython.display import display, Markdown

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged
%store -r restaurants_by_4m_coverage
%store -r time_differences
%store -r time_differences_details
%store -r before_after_details_true

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# Time Differences
%store -r restaurant_data_unprocessed
timezones_acronyms = {}
for loc_id, df in restaurant_data_unprocessed.items():
    time = df['created_at'].iloc[0]
    timezone = time.strip('0123456789-+: ')
    timezones_acronyms[loc_id] = timezone
timezones = {
    '0RJH3FFPYBPEY': 'America/New_York',
    '1SQPTEGYPH0GA': 'America/Denver',
    '3AXDVZJYN9DRS': 'Europe/London',
    '75WYSXR9QBK5M': 'Pacific/Honolulu',
    '78AY09MVJVTYE': 'America/New_York',
    '9XKJD8DQTH559': 'America/New_York',
    'AQD04SM0J92WA': 'America/Los_Angeles',
    'CB2KHY1C2G9PT': 'America/New_York',
    'EMBVNVD207CC6': 'America/New_York',
    'JHDN7CF1C03X5': 'America/Chicago',
    'L3XS7WSJ4AJA3': 'Europe/London',
    'L69HYJ4Y3TR91': 'America/New_York',
    'LBMCPAYT7W36V': 'America/New_York',
    'LBZEEFSBJNB3Z': 'America/Los_Angeles',
    'LFZFT3VASXPED': 'Australia/Sydney',
    'LQ5EH4BKGV61T': 'America/New_York',
    'LZ5MR1TS37E7W': 'America/Los_Angeles',
    'MS8R16DY0JQAM': 'America/Los_Angeles',
    'N0PC58FB2XAZ3': 'America/Chicago',
    'S8MT0YGD2KTN9': 'America/New_York',
    'SAFK7ND1HR6XS': 'America/Los_Angeles',
    'SRQS8F7JWA9MZ': 'America/New_York',
    'V3Q26BHF3SE2H': 'America/New_York',
    'W8T41JZK0ZMEP': 'America/New_York',
    'WJA3YCD4QBWRX': 'America/New_York',
    '1G5AJ17XCH2A8': 'America/Chicago',
    'ADPFRN3QZRCXK': 'America/Los_Angeles',
    'ED5J990H5VAZT': 'America/Los_Angeles',
    '2HRX9P6HKXA8V': 'America/Los_Angeles',
    'C0BE4NDSW26QN': 'America/New_York'
}
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])

before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'] = before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'].str.title()

loc_id = 'L69HYJ4Y3TR91'
df = sales_and_menu_data[loc_id]
df = df.assign(item_modifications = lambda df: df['item_modifications'].str.title())

Read data from parquet files

Time Differences

In [ ]:
df['item_name'].value_counts().size

In [ ]:
food_df = df.query('~dish_category.isin(["Alcohol","Drink","Coffee & Tea","Smoothie"])')

In [ ]:
# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details.loc[loc_id,'cross_over_date'].tz_convert('UTC')

for dish in food_df['item_name'].value_counts().to_frame(name='c').index[:50][::-1]:
    dish_df = food_df.query('item_name == @dish')
    dish_activity = (dish_df['item_quantity']
                                    .resample('W')
                                    .sum()
                                    .to_frame(name='W')
                                    .query('0 < W')
                                    .index
                                    .tz_localize(None)
                                    .to_period('W')
                                    .tolist())

    # For every active week
    for week in dish_activity:

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=loc_id)

# Place a red circle for the promotional item
ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title('Introduction Weekly Activity for Each Dish')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')

# Figure
introduction_fig.tight_layout()

plt.show()

In [ ]:
df['dish_category'].value_counts()